# Primas netas — Serie mensual y variación MoM (solo lectura)

Notebook **solo de lectura**: ejecuta el `SELECT` de `sql/consultas/primas_monto_mensual.sql`
(prima neta mensual por línea de negocio) directamente desde el archivo — así no se duplica
la lógica ni el CASE de `categoria_ramo` — y calcula la **variación mes contra mes (MoM)**
de la prima neta.

Qué hace:

1. Se conecta a Redshift (mismas credenciales locales que el resto de notebooks).
2. Lee y ejecuta el SQL de `sql/consultas/primas_monto_mensual.sql` (últimos 3 años, agrupado por `periodo_fecha` × `categoria_ramo`).
3. Construye la **serie total** de prima neta por mes y su **MoM** (absoluto y %).
4. Construye el **MoM por ramo** (pivot periodo × ramo con su variación %).
5. Exporta `reports/primas_mom.csv` (serie total con MoM) y `reports/primas_mom_por_ramo.csv`.

> `periodo_fecha` es el primer día del mes; el mes en curso aparece con datos parciales,
> por lo que su MoM no es comparable con meses cerrados — se marca como `en_curso`.

## 1) Dependencias y conexión

In [1]:
import json
from pathlib import Path

import pandas as pd
import redshift_connector

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)

# Evitar notacion cientifica en los DataFrames que se muestran:
# montos grandes -> con separador de miles y sin decimales;
# valores pequenos (porcentajes) -> con 2 decimales. Los NaN se dejan como estan.
def _formato_numero(x):
    if pd.isna(x):
        return 'NaN'
    return f'{x:,.0f}' if abs(x) >= 1000 else f'{x:,.2f}'

pd.set_option('display.float_format', _formato_numero)

In [2]:
# Credenciales fuera del repo: notebooks/credenciales_local.py (en .gitignore).
# Si no existe, se piden por consola sin quedar guardadas en ninguna parte.
try:
    from credenciales_local import (
        DEFAULT_HOST, DEFAULT_PORT, DEFAULT_DATABASE, DEFAULT_USER, DEFAULT_PASSWORD
    )
    print('Credenciales cargadas desde credenciales_local.py (archivo local, fuera de git).')
except ImportError:
    import getpass
    print('No se encontro credenciales_local.py - ingresa las credenciales:')
    DEFAULT_HOST = 'corshftanltc-dprogramp.hdicolombia.com.co'
    DEFAULT_PORT = '9519'
    DEFAULT_DATABASE = 'adp_dwh'
    DEFAULT_USER = input('Usuario Redshift: ')
    DEFAULT_PASSWORD = getpass.getpass('Contrasena Redshift: ')

Credenciales cargadas desde credenciales_local.py (archivo local, fuera de git).


In [3]:
conn = redshift_connector.connect(
    host=DEFAULT_HOST,
    port=int(DEFAULT_PORT),
    database=DEFAULT_DATABASE,
    user=DEFAULT_USER,
    password=DEFAULT_PASSWORD,
)
cursor = conn.cursor()
print('Conectado (solo lectura).')


def run_query(sql):
    """Ejecuta un SELECT (solo lectura) y devuelve un DataFrame."""
    cursor.execute(sql)
    return cursor.fetch_dataframe()

Conectado (solo lectura).


## 2) Ejecutar el SQL de `sql/consultas/primas_monto_mensual.sql`

Se lee el archivo tal cual (fuente única de la lógica de prima neta por ramo) y se ejecuta.

In [4]:
RUTA_SQL = Path('..') / 'sql' / 'consultas' / 'primas_monto_mensual.sql'
sql_07 = RUTA_SQL.read_text(encoding='utf-8')
print(f'Ejecutando {RUTA_SQL} ({len(sql_07)} caracteres)...')

df = run_query(sql_07)
df['periodo_fecha'] = pd.to_datetime(df['periodo_fecha'])
df['prima_neta'] = pd.to_numeric(df['prima_neta'])
print(f'Filas: {len(df)} | Periodos: {df["periodo_fecha"].nunique()} | Ramos: {df["categoria_ramo"].nunique()}')
df.head(12)

Ejecutando ..\sql\07_primas_monto_mensual.sql (5178 caracteres)...
Filas: 610 | Periodos: 37 | Ramos: 18


,periodo_fecha,categoria_ramo,prima_neta
0,2026-08-01,Daños Materiales,"389,913,336"
1,2026-08-01,Otros,"181,075,584"
2,2026-08-01,Vida,"9,589,288,580"
3,2026-08-01,Exequias,"19,014,484"
4,2026-08-01,Asistencias,"116,005,034"
5,2026-08-01,Transporte / Navegación,"262,373,200"
6,2026-08-01,Responsabilidad Civil,"78,638,401"
7,2026-08-01,Salud,"2,123,822,944"
8,2026-08-01,Accidentes Personales,"14,539,327"
9,2026-08-01,Cumplimiento,"69,214,643"


## 3) Serie total de prima neta + MoM

Prima neta total del mes (suma de todos los ramos) y su variación contra el mes inmediatamente
anterior — absoluta y porcentual. El último mes se marca `en_curso` (datos parciales).

In [5]:
# Total por mes (todos los ramos)
serie = (df.groupby('periodo_fecha', as_index=False)['prima_neta']
           .sum()
           .sort_values('periodo_fecha')
           .reset_index(drop=True))

serie['periodo'] = serie['periodo_fecha'].dt.strftime('%Y-%m')
serie['var_abs_mom'] = serie['prima_neta'].diff()
serie['var_pct_mom'] = serie['prima_neta'].pct_change() * 100

# El mes mas reciente presente en la fuente = mes en curso (parcial)
mes_en_curso = serie['periodo_fecha'].max()
serie['en_curso'] = serie['periodo_fecha'].eq(mes_en_curso)

serie_mostrar = serie[['periodo', 'prima_neta', 'var_abs_mom', 'var_pct_mom', 'en_curso']].copy()
serie_mostrar

,periodo,prima_neta,var_abs_mom,var_pct_mom,en_curso
0,2023-08,"90,244,147,327",NaN,NaN,False
1,2023-09,"91,516,610,290","1,272,462,962",1.41,False
2,2023-10,"95,374,417,253","3,857,806,964",4.22,False
3,2023-11,"107,363,482,389","11,989,065,136",12.57,False
4,2023-12,"112,077,173,716","4,713,691,327",4.39,False
5,2024-01,"106,094,219,285","-5,982,954,431",-5.34,False
6,2024-02,"94,841,966,031","-11,252,253,254",-10.61,False
7,2024-03,"94,851,988,840","10,022,809",0.01,False
8,2024-04,"96,057,514,641","1,205,525,800",1.27,False
9,2024-05,"101,068,739,145","5,011,224,505",5.22,False


In [6]:
# Vista compacta de los ultimos 12 meses con formato legible
ult = serie.tail(12).copy()
def _fmt_money(x):
    return '' if pd.isna(x) else f'{x:,.0f}'
def _fmt_pct(x):
    return '' if pd.isna(x) else f'{x:+.1f}%'
tabla = pd.DataFrame({
    'Periodo': ult['periodo'],
    'Prima neta': ult['prima_neta'].map(_fmt_money),
    'Var. MoM ($)': ult['var_abs_mom'].map(_fmt_money),
    'Var. MoM (%)': ult['var_pct_mom'].map(_fmt_pct),
    'En curso': ult['en_curso'].map({True: 'si', False: ''}),
})
print('Ultimos 12 meses — prima neta total y variacion mes contra mes:')
tabla.to_string(index=False)
print(tabla.to_string(index=False))

Ultimos 12 meses — prima neta total y variacion mes contra mes:
Periodo      Prima neta    Var. MoM ($) Var. MoM (%) En curso
2025-09 120,192,760,810  -1,045,746,015        -0.9%         
2025-10 110,703,381,415  -9,489,379,395        -7.9%         
2025-11 123,178,654,163  12,475,272,748       +11.3%         
2025-12 121,510,763,202  -1,667,890,961        -1.4%         
2026-01 116,363,028,660  -5,147,734,542        -4.2%         
2026-02 132,842,449,516  16,479,420,856       +14.2%         
2026-03 134,453,148,744   1,610,699,227        +1.2%         
2026-04 126,748,713,138  -7,704,435,606        -5.7%         
2026-05 128,481,738,415   1,733,025,278        +1.4%         
2026-06 144,546,956,166  16,065,217,751       +12.5%         
2026-07 137,568,040,448  -6,978,915,718        -4.8%         
2026-08  54,078,637,906 -83,489,402,543       -60.7%       si


### Gráfico rápido (opcional)

Si `matplotlib` está instalado, muestra la serie total y las barras de MoM %.

In [7]:
try:
    import matplotlib.pyplot as plt
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
    ax1.plot(serie['periodo_fecha'], serie['prima_neta'], color='#0f7a3c', marker='o', ms=3)
    ax1.set_title('Prima neta total por mes')
    ax1.grid(alpha=.3)
    colores = ['#8dc63f' if v >= 0 else '#c0392b' for v in serie['var_pct_mom'].fillna(0)]
    ax2.bar(serie['periodo_fecha'], serie['var_pct_mom'], width=20, color=colores)
    ax2.axhline(0, color='#333', lw=.8)
    ax2.set_title('Variacion MoM (%)')
    ax2.grid(alpha=.3)
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib no instalado — se omite el grafico (las tablas y CSV siguen disponibles).')

matplotlib no instalado — se omite el grafico (las tablas y CSV siguen disponibles).


## 4) MoM por ramo (línea de negocio)

Pivot `periodo × categoria_ramo` con la prima neta, y la variación % mes contra mes de cada ramo.

In [8]:
pivot = (df.pivot_table(index='periodo_fecha', columns='categoria_ramo',
                        values='prima_neta', aggfunc='sum')
           .sort_index())

# Variacion % MoM por ramo
pivot_mom = pivot.pct_change() * 100

# Tabla larga: periodo, ramo, prima_neta, var_pct_mom
largo = (pivot.reset_index()
              .melt(id_vars='periodo_fecha', var_name='categoria_ramo', value_name='prima_neta'))
largo_mom = (pivot_mom.reset_index()
                  .melt(id_vars='periodo_fecha', var_name='categoria_ramo', value_name='var_pct_mom'))
por_ramo = largo.merge(largo_mom, on=['periodo_fecha', 'categoria_ramo'])
por_ramo = por_ramo.dropna(subset=['prima_neta']).sort_values(['periodo_fecha', 'categoria_ramo'])
por_ramo['periodo'] = por_ramo['periodo_fecha'].dt.strftime('%Y-%m')

# MoM del ultimo mes cerrado (o en curso) por ramo, ordenado por magnitud
ultimo = por_ramo[por_ramo['periodo_fecha'].eq(mes_en_curso)].copy()
print(f'Variacion MoM por ramo — {mes_en_curso.strftime("%Y-%m")} (mes en curso, parcial):')
ultimo[['categoria_ramo', 'prima_neta', 'var_pct_mom']].sort_values('var_pct_mom', ascending=False)

Variacion MoM por ramo — 2026-08 (mes en curso, parcial):


,categoria_ramo,prima_neta,var_pct_mom
184,Aviación,"2,942,000","73,549,900"
591,Salud,"2,123,822,944",-37.41
147,Autos,"39,834,102,058",-48.49
110,Asistencias,"116,005,034",-66.18
665,Vida,"9,589,288,580",-68.29
628,Transporte / Navegación,"262,373,200",-73.67
221,Cumplimiento,"69,214,643",-87.57
480,PYME / Multiriesgo,"963,622,201",-89.93
258,Daños Materiales,"389,913,336",-90.41
332,Hogar,"345,045,713",-91.06


## 5) Exportar CSVs a `reports/`

In [9]:
reports = Path('..') / 'reports'
reports.mkdir(exist_ok=True)

salida_total = serie[['periodo', 'periodo_fecha', 'prima_neta', 'var_abs_mom', 'var_pct_mom', 'en_curso']]
salida_total.to_csv(reports / 'primas_mom.csv', index=False, encoding='utf-8-sig')

salida_ramo = por_ramo[['periodo', 'periodo_fecha', 'categoria_ramo', 'prima_neta', 'var_pct_mom']]
salida_ramo.to_csv(reports / 'primas_mom_por_ramo.csv', index=False, encoding='utf-8-sig')

print('Guardado:')
print(' -', (reports / 'primas_mom.csv').resolve())
print(' -', (reports / 'primas_mom_por_ramo.csv').resolve())

Guardado:
 - C:\Users\Wilson.Jerez\OneDrive - HDI Seguros\Documentos\hdi-primas-data-quality\reports\primas_mom.csv
 - C:\Users\Wilson.Jerez\OneDrive - HDI Seguros\Documentos\hdi-primas-data-quality\reports\primas_mom_por_ramo.csv
